In [10]:
from openrouter_client import *
from roboflow_client import *

import dotenv
import os
import json
import time
import tqdm
import base64
from io import BytesIO
from PIL import Image
import requests
from tqdm.notebook import tqdm as tqdmnote
import openai as opi

from threading import Thread, Event

for key in os.environ.keys():
    os.environ.pop(key)

dotenv.load_dotenv()

True

In [11]:
rf_client = RoboflowAPIClient(os.getenv("ROBOFLOW_API_KEY"), os.getenv("ROBOFLOW_WORKSPACE_ID"), os.getenv("ROBOFLOW_PROJECT_ID"))
or_client = OpenRouterClient(os.getenv("OPENROUTER_API_KEY"))
print(os.getenv("ROBOFLOW_PROJECT_ID"))

veri-dataset


In [34]:
# Load the dense caption
with open("./our-dataset-captions.json", "r", encoding="utf-8") as file:
    caption_data = json.load(file)

print(len(caption_data.keys()))

345


In [29]:
# Get the annotations from the captions instead of images

# For each model
# For each image
# For a prompt set, get model output for each and daisy chain

# Can parallelize per model

"""
PROMPT: You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Assume the description is from the point of view of a person physically present in the scene. Hazard: A scene is hazardous if normal interaction would likely cause immediate physical harm, and the danger is unavoidable or easy to miss. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present. Before predicting the code, internally reason through these steps. (1) Identify any real physically present element that could cause immediate physical harm without speculating. (2) If harm is likely and real, determine whether it is unavoidable or easy to miss. (3) Independently determine whether any element violates normal expectations for this environment. (4) Decide whether anomaly and hazard are present independently. You MUST obey this strict output format: <a brief, explicit justification for the final decision>: <code> Where <code> is ONLY one of: 00 Safe 10 Anomalous 01 = Hazardous 11 = Anomalous Hazardous - No extra paragraphs, no bullet points, no additional commentary.
"""

def query_model(model: str, img_data, prompt_set, output_dat):
    if len(prompt_set) == 0:
        print("No prompt!")
        return ""
    # unpack the img data
    
    or_client.prompt = "You are a helpful assistant"
    model_prediction = ""
    model_response = ""
    caption = img_data["caption"]
    img_id = img_data["id"]
    img_url = img_data["url"]
    img_label = img_data["label"]

    history = ""
    # for each prompt in the set, takes any past response, appends it, and asks with next prompt
    # Kind of self-generating few shot prompt
    for prompt in prompt_set:
        history += prompt + "\n" + caption if not history else "\n" + prompt
        model_prediction, model_response = or_client.classify_caption(history)
        history += "\n" + model_response

    output_dat.append({
        "image_id": img_id,
        "image_url": img_url,
        "ground_truth": img_label,
        "model_prediction": model_prediction,
        "model_response": model_response,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "match": model_prediction == img_label
    })

stop_model_thread = Event()
def run_model(model, data, prompt_set, output, class_limit = None):
    print(f"Running model {model}")
    run_outputs = []
    limits = {}
    try:
        bar = tqdmnote(data)
        for i, image in enumerate(bar):
            if(stop_model_thread.is_set()): break

            sample = data[image]
            label = sample["label"]
            if class_limit and label in limits and limits[label] >= class_limit:
                print("Reached limit for class", label)
                continue
            limits[label] = 1 if label not in limits else limits[label] + 1 
            sample["id"] = image
            query_model(model, sample, prompt_set, run_outputs)
    except:
        pass # just move on and output current results
    output[model] = {"prompts": prompt_set, "runs": run_outputs}


In [32]:
outputs = {}
prompt_set2 = [
        "Think about the description and what potential sources of hazards and anomalies are and how hazard might differ from anomaly",
        "Use your thoughts to identify any hazards in the scene. A hazard is anything that can cause harm to people or property and it is unavoidable to easy to miss in the description.",
        "Seperately, think about anomalies in the description. Anomaly is anything that defies common sense and sources might include objects that are not inline with the context of the image.",
        "Decide if the description is safe, anomalous, hazardous, or both using your thoughts. You MUST obey this strict output format: <a brief, explicit justification for the final decision>: <code> Where <code> is ONLY one of: 00 Safe 10 Anomalous 01 = Hazardous 11 = Anomalous Hazardous - No extra paragraphs, no bullet points, no additional commentary."
]
cot_prompts = [
    """You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present. Be concise and respond to the following:

    In adcordance with the definition for "Hazard," Identify any real physically present element that could cause immediate harm. Also consider if people or property are directly in danger.
    """,
    """
    Independently determine whether any element violates normal expectations for this environment using the definition for "Anomaly".
    """,
    """
    Decide whether anomaly and hazard are present independently. You MUST obey this strict output format: <a brief, explicit justification for the final decision>: <code> Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
    """
]
models = ["x-ai/grok-4.1-fast", "google/gemini-3-flash-preview", "qwen/qwen3.5-122b-a10b"]

# Chain of though run

In [31]:

threads = [Thread(target=run_model, args=(model, caption_data, cot_prompts, outputs, 1)) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    while(True): time.sleep(1)
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()
    


# Save the file
for model in outputs:
    output_dir = f"./results/{model.replace("/", "_")}_run"
    i = 0;
    while(os.path.isdir(output_dir + f"{i:03d}")):
        i += 1
    output_dir +=  f"{i:03d}"
    os.mkdir(output_dir)
    with open(f"{output_dir}/results.json", "w", encoding="utf-8") as file:
        json.dump(outputs[model], file, indent=4)

Running model x-ai/grok-4.1-fastRunning model google/gemini-3-flash-preview



  0%|          | 0/345 [00:00<?, ?it/s]

  0%|          | 0/345 [00:00<?, ?it/s]

Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
Reached limit for class 00
R

# Few Shot Run

In [ ]:
threads = [Thread(target=run_model, args=(model, caption_data, prompt_set, outputs)) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    while(True): time.sleep(1)
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()
    


# Save the file
for model in outputs:
    output_dir = f"./results/{model.replace("/", "_")}_run"
    i = 0;
    while(os.path.isdir(output_dir + f"{i:03d}")):
        i += 1
    output_dir +=  f"{i:03d}"
    os.mkdir(output_dir)
    with open(f"{output_dir}/results.json", "w", encoding="utf-8") as file:
        json.dump(outputs[model], file, indent=4)